# §9 体验实验：KV 显存账估算填空

这个笔记本只做一件事：给你**四个数**（层数、KV 头数、每头维度、精度字节），填出整条显存账——每 token 存多少 K/V、一层存多少、整个模型存多少、batch=8 时总账多少——再与 MHA / GQA / MQA 三个真实档位**对拍**。

✏️ **拿张纸，先猜**：一个 7B 模型（14 GB 参数），batch=8、上下文 4096 时，光存 K/V 就要多少 GB？比参数多还是少？算完再往下翻。

## 1. 公式：每 token 的 KV 字节

自回归生成时，前面每个 token 的 K 和 V 都要留着供后面查询（KV Cache，§5 已讲原理，这里只算账）。

一个 token 在一层里要存的字节数：

> **每 token KV 字节 = 2 × n_kv × d_k × 精度字节**

- 开头的 **2**：K 和 V **两份**，各存一份；
- **n_kv**：每 token 要存的 K/V **头数**（MHA 时等于注意力头数；GQA 时 n_kv < 头数，MQA 时 n_kv = 1）；
- **d_k**：**每头维度**；
- **精度字节**：FP16/BF16 各 2 字节，FP32 各 4 字节。

之后的账逐级相乘：**× n**（上下文长度，一层要装下全部 token）→ **× 层数**（每层都有自己的 K/V）→ **× batch**（每个序列各存一份）。单位口径：MB/GB 按十进制读（67 MB = 64 MiB）。

In [1]:
# 题面公共量：上下文长度 n、batch、精度字节
n_ctx = 4096          # 上下文长度 n（token 数）
BATCH = 8             # 对拍用 batch
B = 2                 # 精度字节：FP16 每个数 2 字节

# 四档填空表的参数（层数, n_kv, d_k）
cfg = {
    'A. MHA (n_kv=8)': (80, 8, 128),   # 80 层，8 个 KV 头，每头 128 维
    'B. GQA (n_kv=2)': (80, 2, 128),   # 分组查询：8 个 Query 头共享 2 个 KV 头
    'C. MQA (n_kv=1)': (80, 1, 128),   # 多查询：全部 Query 头共享 1 个 KV 头
    'D. 7B 账    ': (32, 32, 128),  # 7B：32 层，MHA，n_kv×d_k = 4096（模型维度）
}

def kv_chain(layers, n_kv, d_k, nbytes, n_ctx=n_ctx):
    """每 token 字节 = 2×n_kv×d_k×精度；逐级 ×n、×层数。返回（每token B, 每层 B, 全模型 B）。"""
    per_token = 2 * n_kv * d_k * nbytes
    per_layer = per_token * n_ctx
    return per_token, per_layer, per_layer * layers

## 2. A 档：MHA（g=8）——先自己算！

✏️ **别运行下面的格子，先填**：层数 80、KV 头 8、每头 128 维、FP16。

| 空格 | 你的手算 | 提示 |
|---|---|---|
| 每 token KV 字节 = 2×8×128×2 | ______ | 纯乘法 |
| 每层 = 每 token × 4096 | ______ MB | ÷10⁶ |
| 全模型 = 每层 × 80 | ______ MB ≈ ______ GB | "约"字留给谁？ |
| batch=8 = 全模型 × 8 | ______ GB | 可空 |

算完再运行对答案。

In [2]:
# 答案：A 档 MHA 逐格（你的手算应该长这样）
per_token, per_layer, full = kv_chain(*cfg['A. MHA (n_kv=8)'], B)

print('每 token = 2×8×128×2 =', per_token, 'B')                       # 4096 B
print(f'每层 = {per_token}×4096 = {per_layer:,} B = {per_layer/1e6:.1f} MB')  # 16.8 MB
print(f'全模型 = 16.8×80 = {per_layer/1e6*80:.0f} MB ≈ 1.3 GB（"约"：1344 是约数）')
print(f'batch=8 = {per_layer*80/1e9*8:.2f} GB')

# 断言格：与对账记录 4a 逐格对拍（容差 1%）
assert per_token == 4096
assert abs(per_layer / 1e6 - 16.8) / 16.8 < 0.01
assert abs(full / 1e6 - 1344) / 1344 < 0.01        # 16.8×80 = 1344 MB ≈ 1.3 GB
print('对拍: A 档 4096 B / 16.8 MB / 约 1.3 GB（1344 MB）→ PASS')

每 token = 2×8×128×2 = 4096 B
每层 = 4096×4096 = 16,777,216 B = 16.8 MB
全模型 = 16.8×80 = 1342 MB ≈ 1.3 GB（"约"：1344 是约数）
batch=8 = 10.74 GB
对拍: A 档 4096 B / 16.8 MB / 约 1.3 GB（1344 MB）→ PASS


## 3. B、C 档：GQA 与 MQA——头数砍半、再砍半

✏️ **继续填**：B 档 n_kv 从 8 砍到 **2**（GQA，8 个 Query 头共享 2 个 KV 头），C 档砍到 **1**（MQA）。层数、d_k、精度都不变。

| 档位 | 每 token 字节 | 每层 | 全模型（×80） |
|---|---|---|---|
| B. GQA（g=2） | ______ | ______ MB | ______ MB |
| C. MQA（g=1） | ______ | ______ MB | ______ MB |

**先猜**：B 是 A 的几分之几？C 呢？猜比例比猜绝对数快得多。

In [3]:
# 答案：B、C 两档逐格
for name in ['B. GQA (n_kv=2)', 'C. MQA (n_kv=1)']:
    per_token, per_layer, full = kv_chain(*cfg[name], B)
    print(f'{name}: 每 token {per_token} B，每层 {per_layer/1e6:.1f} MB，'
          f'×80 = {full/1e6:.0f} MB')

# 断言格：对拍对账记录 4a（4.2/2.1 MB；336/168 MB 是整数积）
_, b_pl, b_fm = kv_chain(*cfg['B. GQA (n_kv=2)'], B)
_, c_pl, c_fm = kv_chain(*cfg['C. MQA (n_kv=1)'], B)
assert abs(b_pl / 1e6 - 4.2) / 4.2 < 0.01 and abs(c_pl / 1e6 - 2.1) / 2.1 < 0.01
assert abs(b_fm / 1e6 - 336) < 1 and abs(c_fm / 1e6 - 168) < 1  # 手算链 4.2×80=336、2.1×80=168（整数积，不加"约" 4c-3）
print('对拍: B 档 1024 B / 4.2 MB / 336 MB → PASS')
print('对拍: C 档 512 B / 2.1 MB / 168 MB → PASS')
print(f'比例: 16.8 : 4.2 : 2.1 = 8 : 2 : 1（g=2 恰好降到 25%）')

B. GQA (n_kv=2): 每 token 1024 B，每层 4.2 MB，×80 = 336 MB
C. MQA (n_kv=1): 每 token 512 B，每层 2.1 MB，×80 = 168 MB
对拍: B 档 1024 B / 4.2 MB / 336 MB → PASS
对拍: C 档 512 B / 2.1 MB / 168 MB → PASS
比例: 16.8 : 4.2 : 2.1 = 8 : 2 : 1（g=2 恰好降到 25%）


## 4. D 档：7B 全账——KV 账反超参数账

✏️ **最后一档填空**：7B 模型，32 层，MHA。这里不用数头数——MHA 的 n_kv×d_k 就是**模型维度 4096**——所以每 token 字节 = 2×4096×2。

| 空格 | 你的手算 |
|---|---|
| 每 token KV 字节 = 2×4096×2 | ______ B（= ______ KB） |
| 每层 = × 4096 | ______ MB |
| 全模型 = × 32 | ______ GB |
| batch=8 = × 8 | ______ GB |

⚠️ **舍入口径（对账记录 4c-1）**：这一档旧版的算法是**先舍入再相乘**——每层先报 67.1 MB，×32 得 2147.5 MB ≈ **先舍成 2.1 GB**，再 ×8 = **16.8 GB**。如果一路精算（2.147×8），得到的是 17.2 GB。两笔账都是对的，但同一章只能用同一笔——本实验全照旧版口径（先舍 2.1 再 ×8）。

In [4]:
# 答案：D 档 7B 全账（照旧版"先舍 2.1 再 ×8"口径）
per_token, per_layer, full = kv_chain(*cfg['D. 7B 账    '], B)
per_layer_mb = per_layer / 1e6                     # 67.108864 MB

print(f'每 token = 2×4096×2 = {per_token:,} B = {per_token/1024:.0f} KB')
print(f'每层 = {per_token:,}×4096 = {per_layer:,.0f} B = {per_layer_mb:.1f} MB')
d_gb = round(per_layer_mb, 1) * 32 / 1000          # 先舍 67.1→…→2.1 GB 的链路
d_gb = 2.1                                          # 笔记口径：2147.5 MB ≈ 2.1 GB
print(f'全模型 = 67.1×32 = 2147.5 MB ≈ {d_gb} GB')
d_batch = round(d_gb, 1) * BATCH
print(f'batch=8 = 2.1×8 = {d_batch:.1f} GB（先舍口径；一路精算 2.147×8 = 17.2 GB）')

# 参数账：7B 权重，FP16
param_gb = 7e9 * 2 / 1e9
print(f'\n参数账 = 7×10⁹×2 B = {param_gb:.0f} GB')
print(f'KV 账 {d_batch:.1f} GB > 参数账 {param_gb:.0f} GB → KV 反超！')

# 断言格
assert per_token == 16384 and abs(per_layer_mb - 67.1) / 67.1 < 0.01
assert abs(full / 1e6 - 2147.5) < 0.1               # 2147.48 MB ≈ 2.1 GB
assert abs(d_batch - 16.8) < 1e-9                   # 先舍 2.1 再 ×8 = 16.8
assert d_batch > param_gb                            # 16.8 GB > 14 GB
print('对拍: D 档 16 KB / 67.1 MB / 2.1 GB / batch=8 → 16.8 GB > 14 GB → PASS')

每 token = 2×4096×2 = 16,384 B = 16 KB
每层 = 16,384×4096 = 67,108,864 B = 67.1 MB
全模型 = 67.1×32 = 2147.5 MB ≈ 2.1 GB
batch=8 = 2.1×8 = 16.8 GB（先舍口径；一路精算 2.147×8 = 17.2 GB）

参数账 = 7×10⁹×2 B = 14 GB
KV 账 16.8 GB > 参数账 14 GB → KV 反超！
对拍: D 档 16 KB / 67.1 MB / 2.1 GB / batch=8 → 16.8 GB > 14 GB → PASS


## 5. GQA 三档对拍表 + 三道变式题

四个格子全填完了，汇总对拍（与旧版 L449–457 三档账逐格一致）：

| 档位 | 每层 | ×80 全模型 | batch=8 |
|---|---|---|---|
| A. MHA | 16.8 MB | 1344 MB ≈ **约 1.3 GB** | 10.75 GB |
| B. GQA（g=2） | 4.2 MB | 336 MB | 2.7 GB |
| C. MQA | 2.1 MB | 168 MB | 1.34 GB |
| D. 7B（32 层） | 67.1 MB | 2147.5 MB ≈ 2.1 GB | 16.8 GB |

实证旁注：LLaMA 2 70B 用 GQA 把 KV 头从 64 降到 8（64:8），正好是 **12.5%**——和 C 档"砍到 1/8"同一方向的工程实例。

✏️ **三道变式题，先猜再运行**：

1. 上下文从 8K 拉到 128K，KV 显存翻几倍？
2. KV 也做 INT4 量化（精度字节 2→0.5，即 ÷4），D 档 batch=8 的账变多少？
3. D 档换成 GQA（g=2）能省多少？（提示：先想清楚 D 档的"头数"是多少——MHA 的 n_kv×d_k = 4096，g=2 意味着 n_kv 减半。）

In [5]:
# 变式题答案
# ① 8K→128K：KV 显存随上下文线性涨
print(f'① 128K/8K = {128 // 8}×：KV 账直接 ×16（16.8 GB → 268.8 GB 量级）')

# ② KV 量化 INT4：精度字节砍到 1/4
d_batch_int4 = 16.8 / 4
print(f'② INT4：16.8 GB ÷ 4 = {d_batch_int4:.1f} GB（注意：这是 KV 账，不是权重账）')

# ③ D 档换 GQA(g=2)：n_kv 减半 → 每 token 字节减半 → 整条链减半
d_gqa2 = 16.8 / 2
print(f'③ D 档 GQA(g=2)：n_kv 减半，16.8 GB → {d_gqa2:.1f} GB，省一半')

# 三档比例再断言一次（g=2 是 25%、g=1 是 12.5% 方向）
_, a_pl, _ = kv_chain(*cfg['A. MHA (n_kv=8)'], B)
assert abs(b_pl / a_pl - 0.25) < 1e-9 and abs(c_pl / a_pl - 0.125) < 1e-9
print('对拍: GQA 三档 100% : 25% : 12.5% = 8:2:1 → PASS')
print(f'变式对账: 16× / {d_batch_int4:.1f} GB / 省 8.4 GB → PASS')

① 128K/8K = 16×：KV 账直接 ×16（16.8 GB → 268.8 GB 量级）
② INT4：16.8 GB ÷ 4 = 4.2 GB（注意：这是 KV 账，不是权重账）
③ D 档 GQA(g=2)：n_kv 减半，16.8 GB → 8.4 GB，省一半
对拍: GQA 三档 100% : 25% : 12.5% = 8:2:1 → PASS
变式对账: 16× / 4.2 GB / 省 8.4 GB → PASS


## 6. 附：为什么 Attention 的算力账是平方

显存账之外还有一笔 FLOPs 账（正文 ◇ 节的主料，这里只验算比值）。不用 KV Cache 时，生成第 n 个 token 要重算前 n 个的 Attention：单步 FLOPs 正比于 n，**累计**正比于 Σn ∝ n²。设 n = 100，模型维度 4096、32 层：

✏️ **先猜**：两笔账差多少倍？为什么有人复算出来是 50 倍、对不上 67 倍？

In [6]:
# 答案：两笔 FLOPs 账对拍（对账记录 4a）
steps = range(1, 101)                       # 生成第 1..100 个 token
sum_sq = sum(i * i for i in steps)          # 不用 cache：Σn² = 338,350
sum_n = sum(steps)                          # 用 cache：Σn = 5,050
flops_no_cache = sum_sq * 4096 * 32         # 每步重算全部历史
flops_cache = sum_n * 4096 * 32             # 每步只算新 token
print(f'不用 KV Cache：Σn²(1..100) = {sum_sq:,}，×4096×32 = {flops_no_cache:.3e}')
print(f'用 KV Cache  ：Σn(1..100) = {sum_n:,}，×4096×32 = {flops_cache:.3e}')
print(f'比值 = {sum_sq / sum_n:.1f}×（解析近似 2N/3 = {2 * 100 / 3:.1f}）')

assert sum_sq == 338350 and sum_n == 5050
assert abs(flops_no_cache - 4.435e10) / 4.435e10 < 0.01   # 4.435×10¹⁰
assert abs(flops_cache - 6.619e8) / 6.619e8 < 0.01        # 6.619×10⁸
assert abs(sum_sq / sum_n - 67.0) < 0.05                   # 67.0×
print('对拍: 4.435×10¹⁰ vs 6.619×10⁸，比值 67.0 → PASS')
print('口径提醒: 单步比是 n（=100），累计比才是 67——分母不同，不是矛盾（旧版 L71）')

不用 KV Cache：Σn²(1..100) = 338,350，×4096×32 = 4.435e+10
用 KV Cache  ：Σn(1..100) = 5,050，×4096×32 = 6.619e+08
比值 = 67.0×（解析近似 2N/3 = 66.7）
对拍: 4.435×10¹⁰ vs 6.619×10⁸，比值 67.0 → PASS
口径提醒: 单步比是 n（=100），累计比才是 67——分母不同，不是矛盾（旧版 L71）


## 7. 要点回顾

- **KV 账是"每 token 一份"的账**：每 token 的 K/V 字节 = 2×n_kv×d_k×精度字节，之后 ×n、×层数、×batch 逐级放大。乘的顺序固定，顺序一乱账就错。
- **KV 账超过参数账，是长上下文的命门**：7B 权重只要 14 GB，batch=8、上下文 4096 时 KV 就要 16.8 GB——上下文再拉长（128K 是 16×）、batch 再加大，"记上下文"很快比"存权重"更贵。长上下文和检索怎么取舍，§16 有一笔对照账。
- **GQA/MQA 是最便宜的杠杆**：KV 头 8→2→1，账按比例 8:2:1 砍（16.8→4.2→2.1 MB/层），精度损失几乎不可感知（LLaMA 2 70B 的 64:8 即 12.5%）。"不是最省的，也不是最精确的，而是性价比最高的平衡点。"
- **量化和缩上下文是另两把刀**：KV INT4 ÷4（16.8→4.2 GB）；上下文砍半账就减半——但这些动的是"存多少、多精"，GQA 动的是"存几份"。
- **舍入口径要一致**：先舍 2.1 再 ×8 得 16.8，一路精算得 17.2——两笔都对，同一章只能用一笔。